# Notebook 02 — Crop Truth Annotation

**Frozen scope (plan §12.3):**
- Annotation interface
- Annotation export validation

**This notebook does NOT:**
- Run any detectors
- Perform truth matching
- Write candidate tables

**This notebook DOES:**
- Clears all prior annotation data on every run (fresh start)
- Loads the crop registry produced by Notebook 01
- Presents each crop as a 4-panel interactive figure (all channels + DAPI side by side)
- Clicking any panel places a synced marker on all panels simultaneously
- Collects `TRUE_SPOT` and `UNCERTAIN` clicks, applies local-max refinement
- Stores both raw and refined coordinates in crop-local and well frames (§2.1)
- Assigns deterministic `annotation_id` per click (§2.2)
- Exports validated `annotations.parquet` matching the §3.6 schema
- Saves per-crop annotation overlays to `artifacts/galleries/`

**Labels (§8.3):**
- `TRUE_SPOT` — left-click
- `UNCERTAIN` — right-click
- Negatives are NOT clicked — derived automatically in Notebook 05

**⚠ Runs fresh every time.** All prior annotation files are deleted at startup.

## 0. Imports and repo discovery

In [ ]:
from __future__ import annotations

import getpass
import hashlib
import json
import shutil
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import tifffile
import yaml

In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / ".git").exists() or (p / "registries").exists():
            return p
    raise RuntimeError("Could not locate repo root.")

REPO_ROOT     = find_repo_root()
NOTEBOOK_NAME = "02_crop_truth_annotation.ipynb"
print("REPO_ROOT =", REPO_ROOT)

## 1. Config

In [ ]:
CFG = {
    # Input
    "crop_registry_subdir": "annotations/crop_registry",
    "tables_subdir":        "tables",

    # Output
    "annotations_subdir":   "annotations/clicked_truth",
    "gallery_subdir":       "artifacts/galleries/annotation_overlays",
    "manifest_subdir":      "artifacts/manifests",

    # Annotator
    "annotator": getpass.getuser(),

    # Registry version
    "annotation_registry_version": "v1",

    # Refinement: snap raw click to nearest local maximum within this radius
    "refinement_policy_id": "local_max_r5",
    "refinement_radius_px": 5,

    # Only annotate crops with these statuses
    "annotate_status_filter": ["pending", "in_progress"],

    # Display contrast percentiles (applied per-channel independently)
    "display_plo": 1,
    "display_phi": 99,

    # Channel display labels — edit to match your actual channel order.
    # Keys are channel_id integers as extracted from filenames by Notebook 01.
    # None = files where channel_id could not be parsed.
    "channel_labels": {
        0:    "DAPI",
        1:    "Ch1",
        2:    "Ch2",
        3:    "Ch3",
    },
}
CFG

## 2. Reset — clear all prior annotation data

In [ ]:
ANNOTATIONS_DIR   = REPO_ROOT / CFG["annotations_subdir"]
GALLERY_DIR       = REPO_ROOT / CFG["gallery_subdir"]
MANIFEST_DIR      = REPO_ROOT / CFG["manifest_subdir"]
TABLES_DIR        = REPO_ROOT / CFG["tables_subdir"]
CROP_REGISTRY_DIR = REPO_ROOT / CFG["crop_registry_subdir"]
canonical_parquet = TABLES_DIR / "annotations.parquet"

if ANNOTATIONS_DIR.exists():
    shutil.rmtree(ANNOTATIONS_DIR)
    print(f"Cleared: {ANNOTATIONS_DIR}")
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)

if GALLERY_DIR.exists():
    shutil.rmtree(GALLERY_DIR)
    print(f"Cleared: {GALLERY_DIR}")
GALLERY_DIR.mkdir(parents=True, exist_ok=True)

if canonical_parquet.exists():
    canonical_parquet.unlink()
    print(f"Cleared: {canonical_parquet}")

for d in [MANIFEST_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Ready for fresh annotation run.")

## 3. Core functions

In [ ]:
def compute_projection(arr, kind="max"):
    arr = np.asarray(arr)
    if arr.ndim == 2:
        return arr.astype(np.float32)
    axes = tuple(range(arr.ndim - 2))
    if kind == "max":
        return np.max(arr, axis=axes).astype(np.float32)
    return np.mean(arr, axis=axes).astype(np.float32)


def percentile_clip(img, lo=1, hi=99):
    lo_v = np.percentile(img, lo)
    hi_v = np.percentile(img, hi)
    return np.clip((img - lo_v) / (hi_v - lo_v + 1e-8), 0, 1)


def crop_to_well(crop_y, crop_x, origin_y, origin_x):
    return crop_y + origin_y, crop_x + origin_x


def refine_to_local_max(img2d, click_y, click_x, radius=5):
    H, W = img2d.shape
    y0 = max(0, int(round(click_y)) - radius)
    y1 = min(H, int(round(click_y)) + radius + 1)
    x0 = max(0, int(round(click_x)) - radius)
    x1 = min(W, int(round(click_x)) + radius + 1)
    patch = img2d[y0:y1, x0:x1]
    if patch.size == 0:
        return click_y, click_x
    dy, dx = np.unravel_index(np.argmax(patch), patch.shape)
    return float(y0 + dy), float(x0 + dx)


def make_annotation_id(dataset_id, well_id, crop_id, annotator,
                       timestamp_utc, raw_well_y, raw_well_x):
    key = "|".join([
        dataset_id, well_id, crop_id, annotator, timestamp_utc,
        f"{raw_well_y:.4f}", f"{raw_well_x:.4f}"
    ])
    return "ann_" + hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]


def build_annotation_record(
    *, dataset_id, well_id, crop_id, annotator, label, confidence,
    raw_click_crop_y_px, raw_click_crop_x_px,
    origin_well_y_px, origin_well_x_px,
    refinement_img2d, refinement_radius_px, refinement_policy_id,
    annotation_registry_version,
):
    timestamp_utc = datetime.now(timezone.utc).isoformat()
    raw_well_y, raw_well_x = crop_to_well(
        raw_click_crop_y_px, raw_click_crop_x_px,
        origin_well_y_px, origin_well_x_px
    )
    ref_crop_y, ref_crop_x = refine_to_local_max(
        refinement_img2d, raw_click_crop_y_px, raw_click_crop_x_px,
        radius=refinement_radius_px
    )
    ref_well_y, ref_well_x = crop_to_well(
        ref_crop_y, ref_crop_x, origin_well_y_px, origin_well_x_px
    )
    annotation_id = make_annotation_id(
        dataset_id, well_id, crop_id, annotator,
        timestamp_utc, raw_well_y, raw_well_x
    )
    return {
        "annotation_id":               annotation_id,
        "dataset_id":                  dataset_id,
        "well_id":                     well_id,
        "crop_id":                     crop_id,
        "annotator":                   annotator,
        "timestamp_utc":               timestamp_utc,
        "label":                       label,
        "confidence":                  confidence,
        "raw_click_crop_y_px":         raw_click_crop_y_px,
        "raw_click_crop_x_px":         raw_click_crop_x_px,
        "raw_click_well_y_px":         raw_well_y,
        "raw_click_well_x_px":         raw_well_x,
        "refined_click_crop_y_px":     ref_crop_y,
        "refined_click_crop_x_px":     ref_crop_x,
        "refined_click_well_y_px":     ref_well_y,
        "refined_click_well_x_px":     ref_well_x,
        "refinement_policy_id":        refinement_policy_id,
        "annotation_registry_version": annotation_registry_version,
    }


REQUIRED_ANNOTATION_COLS = [
    "annotation_id", "dataset_id", "well_id", "crop_id",
    "annotator", "timestamp_utc", "label", "confidence",
    "raw_click_crop_y_px", "raw_click_crop_x_px",
    "raw_click_well_y_px", "raw_click_well_x_px",
    "refined_click_crop_y_px", "refined_click_crop_x_px",
    "refined_click_well_y_px", "refined_click_well_x_px",
    "refinement_policy_id", "annotation_registry_version",
]
VALID_LABELS = {"TRUE_SPOT", "UNCERTAIN"}


def validate_annotations(df):
    missing = [c for c in REQUIRED_ANNOTATION_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    bad = df[~df["label"].isin(VALID_LABELS)]["label"].unique()
    if len(bad):
        raise ValueError(f"Invalid labels: {bad}")
    if df["annotation_id"].duplicated().any():
        raise ValueError("Duplicate annotation_id values")
    coord_cols = [
        "raw_click_crop_y_px", "raw_click_crop_x_px",
        "raw_click_well_y_px", "raw_click_well_x_px",
        "refined_click_crop_y_px", "refined_click_crop_x_px",
        "refined_click_well_y_px", "refined_click_well_x_px",
    ]
    for col in coord_cols:
        if df[col].isna().any():
            raise ValueError(f"NaN in {col}")
    print(f"\u2713 {len(df)} annotations pass schema validation "
          f"({df['label'].value_counts().to_dict()})")

## 4. Load crop registry and image inventory

In [ ]:
# Ensure paths are defined even if reset cell was skipped
CROP_REGISTRY_DIR = REPO_ROOT / CFG["crop_registry_subdir"]
TABLES_DIR        = REPO_ROOT / CFG["tables_subdir"]
ANNOTATIONS_DIR   = REPO_ROOT / CFG["annotations_subdir"]
GALLERY_DIR       = REPO_ROOT / CFG["gallery_subdir"]
MANIFEST_DIR      = REPO_ROOT / CFG["manifest_subdir"]
canonical_parquet = TABLES_DIR / "annotations.parquet"

yaml_files = sorted(CROP_REGISTRY_DIR.glob("*_crop_registry.yaml"))
if not yaml_files:
    raise FileNotFoundError(f"No crop registry YAML in {CROP_REGISTRY_DIR}. Run Notebook 01 first.")
payload = yaml.safe_load(yaml_files[-1].read_text(encoding="utf-8"))
crop_registry_df = pd.DataFrame(payload["records"])
print(f"Loaded {len(crop_registry_df)} crops from {yaml_files[-1].name}")

inv_parquets = sorted(TABLES_DIR.glob("*_image_inventory.parquet"))
if not inv_parquets:
    raise FileNotFoundError(f"No image inventory parquet in {TABLES_DIR}. Run Notebook 01 first.")
inventory_df = pd.read_parquet(inv_parquets[-1])
print(f"Loaded inventory: {len(inventory_df)} files")

print("\nChannels per well:")
display(
    inventory_df.groupby(["well_id", "channel_id"])["file_name"]
    .count().rename("n_files").reset_index()
    .sort_values(["well_id", "channel_id"])
)

In [ ]:
# ── DIAGNOSTIC: inspect what the inventory actually contains ───────────────
# Run this to understand your file structure before the cache is built.
# If channel_id is all None, your filenames don't match the channel regex
# and you need to set channel_id_override below.

print("=== RAW INVENTORY SAMPLE ===")
display(inventory_df[["well_id", "channel_id", "cycle_id", "file_name"]].head(30))

print("\n=== channel_id value counts ===")
print(inventory_df["channel_id"].value_counts(dropna=False).to_string())

print("\n=== Files per (well_id, channel_id) ===")
display(
    inventory_df.groupby(["well_id", "channel_id"], dropna=False)["file_name"]
    .count().rename("n_files").reset_index()
    .sort_values(["well_id", "channel_id"])
)

# If all channel_id are None, inspect a few filenames to understand the naming pattern
if inventory_df["channel_id"].isna().all():
    print("\n*** WARNING: channel_id is None for ALL files. ***")
    print("Sample filenames:")
    for fn in inventory_df["file_name"].head(10):
        print(" ", fn)
    print("\nFull paths (first 5):")
    for fp in inventory_df["file_path"].head(5):
        print(" ", fp)


In [ ]:
# ── CHANNEL ASSIGNMENT FROM WAVELENGTH IN FILENAME ───────────────────────
# Your filenames encode channel as excitation wavelength, not a channel number.
# Examples: 405_nm_Ex -> DAPI, 561_nm_Ex -> Ch561, 638_nm_Ex -> Ch638, BF_nm_Ex -> BF
#
# This cell:
#   1. Parses the wavelength from each filename
#   2. Assigns a stable integer channel_id (0=DAPI/405, 1=561, 2=638, 3=BF)
#   3. Picks one cycle per wavelength (cycle_id==1, or lowest available)
#   4. Updates inventory_df so the cache cell works correctly

import re as _re

# Map wavelength string -> (channel_id int, display label)
# Edit labels here if you want different panel titles
WAVELENGTH_MAP = {
    "405":  (0, "DAPI  (405nm)"),
    "561":  (1, "Ch561 (561nm)"),
    "638":  (2, "Ch638 (638nm)"),
    "BF":   (3, "BF"),
}

def _parse_wavelength(filename):
    """Return (channel_id, label) from filename, or (None, None) if unrecognised."""
    for wl, (ch_id, label) in WAVELENGTH_MAP.items():
        if f"_{wl}_nm_Ex" in filename or f"_{wl}_nm" in filename or f"_{wl}nm" in filename:
            return ch_id, label
        if wl == "BF" and "_BF_" in filename:
            return ch_id, label
    return None, None

# Apply to inventory
parsed = inventory_df["file_name"].apply(_parse_wavelength)
inventory_df["channel_id"]    = [p[0] for p in parsed]
inventory_df["channel_label"] = [p[1] for p in parsed]

# Also update CFG channel_labels to match
CFG["channel_labels"] = {ch_id: label for _, (ch_id, label) in WAVELENGTH_MAP.items()}

# Pick a single preferred cycle — prefer cycle_id==1, else take the lowest
PREFERRED_CYCLE = 1
MULTICHANNEL_MODE = False

print("Updated channel_id counts:")
print(inventory_df["channel_id"].value_counts(dropna=False).to_string())
print()
display(
    inventory_df[["well_id", "channel_id", "channel_label", "cycle_id", "file_name"]]
    .sort_values(["well_id", "channel_id", "cycle_id"])
    .head(20)
)


## 5. Build per-well channel image cache

In [ ]:
# Build channel cache using wavelength-parsed channel_id and channel_label

pending_df = crop_registry_df[
    crop_registry_df["annotator_status"].isin(CFG["annotate_status_filter"])
].reset_index(drop=True)

channel_cache = {}
unique_wells = pending_df[["dataset_id", "well_id"]].drop_duplicates().values.tolist()

for dataset_id, well_id in unique_wells:
    key = (dataset_id, well_id)
    well_files = inventory_df[
        (inventory_df["dataset_id"] == dataset_id) &
        (inventory_df["well_id"] == well_id) &
        (inventory_df["channel_id"].notna())
    ].copy()

    if len(well_files) == 0:
        warnings.warn(f"No files with recognised channel for {well_id}")
        channel_cache[key] = []
        continue

    # For each channel_id pick one file: prefer PREFERRED_CYCLE, else lowest cycle
    chosen_rows = []
    for ch_id, grp in well_files.groupby("channel_id"):
        preferred = grp[grp["cycle_id"] == PREFERRED_CYCLE]
        row = preferred.iloc[0] if len(preferred) else grp.sort_values("cycle_id", na_position="last").iloc[0]
        chosen_rows.append(row)

    channels = []
    for row in chosen_rows:
        ch_id = int(row["channel_id"])
        label = row.get("channel_label", CFG["channel_labels"].get(ch_id, f"Ch{ch_id}"))
        try:
            arr = np.asarray(tifffile.imread(str(row["file_path"])))
            proj = compute_projection(arr, kind="max")
            channels.append({"channel_id": ch_id, "label": label, "projection": proj})
            print(f"  {well_id} ch={ch_id} ({label}) cycle={row['cycle_id']} shape={proj.shape}")
        except Exception as e:
            warnings.warn(f"Skipping {well_id} ch={ch_id}: {e}")

    # Sort by channel_id ascending
    channels.sort(key=lambda c: c["channel_id"])
    channel_cache[key] = channels
    print(f"  -> {len(channels)} channels ready for {well_id}")

print(f"\nCache built for {len(channel_cache)} well(s). {len(pending_df)} crops to annotate.")


## 6. Annotation interface

**Controls:**
- **Left-click** any panel → `TRUE_SPOT` placed on all panels
- **Right-click** any panel → `UNCERTAIN` placed on all panels
- **Middle-click** or **`d` key** → delete nearest click (synced)
- **Close window** → save this crop, move to next

In [ ]:
%matplotlib tk

In [ ]:
LABEL_COLOR = {"TRUE_SPOT": "lime", "UNCERTAIN": "orange"}


def annotate_crop_multichannel(crop_record, channels, cfg):
    dataset_id = crop_record["dataset_id"]
    well_id    = crop_record["well_id"]
    crop_id    = crop_record["crop_id"]
    ymin = int(crop_record["well_ymin_px"])
    xmin = int(crop_record["well_xmin_px"])
    ymax = int(crop_record["well_ymax_px"])
    xmax = int(crop_record["well_xmax_px"])

    n_panels = len(channels)
    if n_panels == 0:
        warnings.warn(f"No channels for {well_id}/{crop_id}, skipping.")
        return []

    # Slice and contrast-stretch each channel
    crop_imgs    = [ch["projection"][ymin:ymax, xmin:xmax] for ch in channels]
    display_imgs = [percentile_clip(img, cfg["display_plo"], cfg["display_phi"])
                    for img in crop_imgs]

    # Refinement uses first non-DAPI channel (channel_id != 0); fall back to ch[0]
    refinement_img = crop_imgs[0]
    for i, ch in enumerate(channels):
        if ch["channel_id"] != 0:
            refinement_img = crop_imgs[i]
            break

    # ── Build figure ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        1, n_panels,
        figsize=(max(4 * n_panels, 12), 7),
        gridspec_kw={"wspace": 0.04}
    )
    axes = [axes] if n_panels == 1 else list(axes)

    for ax, dimg, ch in zip(axes, display_imgs, channels):
        ax.imshow(dimg, cmap="gray", origin="upper")
        ax.set_title(ch["label"], fontsize=11, pad=4)
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(
        f"{well_id}  |  {crop_id[-12:]}  |  y:[{ymin},{ymax})  x:[{xmin},{xmax})\n"
        f"LEFT = TRUE_SPOT     RIGHT = UNCERTAIN     MIDDLE / d = delete     close = next crop",
        fontsize=9, y=1.02
    )
    axes[-1].legend(
        handles=[
            mpatches.Patch(color="lime",   label="TRUE_SPOT"),
            mpatches.Patch(color="orange", label="UNCERTAIN"),
        ],
        loc="upper right", fontsize=8, framealpha=0.7
    )

    # ── Click state ────────────────────────────────────────────────────────
    # Each entry: {y_crop, x_crop, label, artists: [Line2D, ...]}
    clicks = []

    def draw_click(click):
        color = LABEL_COLOR[click["label"]]
        artists = []
        for ax in axes:
            a = ax.plot(
                click["x_crop"], click["y_crop"],
                marker="+", markersize=16, markeredgewidth=2,
                color=color, linestyle="none"
            )[0]
            artists.append(a)
        click["artists"] = artists
        fig.canvas.draw_idle()

    def erase_click(click):
        for a in click.get("artists", []):
            try:
                a.remove()
            except Exception:
                pass
        click["artists"] = []
        fig.canvas.draw_idle()

    def delete_nearest(y_crop, x_crop):
        if not clicks:
            return
        dists = [np.hypot(c["y_crop"] - y_crop, c["x_crop"] - x_crop) for c in clicks]
        idx = int(np.argmin(dists))
        if dists[idx] < 30:
            erase_click(clicks[idx])
            clicks.pop(idx)

    def on_click(event):
        if event.inaxes not in axes:
            return
        x, y = event.xdata, event.ydata
        if x is None or y is None:
            return
        if event.button == 2:
            delete_nearest(y, x)
            return
        label = "TRUE_SPOT" if event.button == 1 else "UNCERTAIN"
        click = {"y_crop": y, "x_crop": x, "label": label, "artists": []}
        clicks.append(click)
        draw_click(click)

    def on_key(event):
        if event.key == "d" and event.inaxes in axes:
            if event.xdata is not None and event.ydata is not None:
                delete_nearest(event.ydata, event.xdata)

    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("key_press_event", on_key)
    plt.tight_layout()
    plt.show(block=True)

    # ── Convert to annotation records ──────────────────────────────────────
    records = []
    for c in clicks:
        rec = build_annotation_record(
            dataset_id=dataset_id, well_id=well_id, crop_id=crop_id,
            annotator=cfg["annotator"],
            label=c["label"],
            confidence=1.0 if c["label"] == "TRUE_SPOT" else 0.5,
            raw_click_crop_y_px=c["y_crop"],
            raw_click_crop_x_px=c["x_crop"],
            origin_well_y_px=ymin, origin_well_x_px=xmin,
            refinement_img2d=refinement_img,
            refinement_radius_px=cfg["refinement_radius_px"],
            refinement_policy_id=cfg["refinement_policy_id"],
            annotation_registry_version=cfg["annotation_registry_version"],
        )
        records.append(rec)

    plt.close(fig)
    return records

## 7. Annotation loop

In [ ]:
RUN_ID       = f"nb02_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
RUN_START_TS = time.time()
all_records  = []
session_log  = []

print(f"Session  : {RUN_ID}")
print(f"Annotator: {CFG['annotator']}")
print(f"Crops    : {len(pending_df)}")
print("-" * 60)

for i, row in pending_df.iterrows():
    crop_record = row.to_dict()
    crop_id    = crop_record["crop_id"]
    dataset_id = crop_record["dataset_id"]
    well_id    = crop_record["well_id"]
    channels   = channel_cache.get((dataset_id, well_id), [])

    if not channels:
        warnings.warn(f"No channels for {well_id}, skipping {crop_id}")
        continue

    print(f"\n[{i+1}/{len(pending_df)}] {well_id} | {crop_id[-12:]} "
          f"| {len(channels)} channels | tags={crop_record['selection_tags']}")

    t0 = time.time()
    try:
        records = annotate_crop_multichannel(crop_record, channels, CFG)
    except Exception as e:
        warnings.warn(f"Error on {crop_id}: {e}")
        session_log.append({"crop_id": crop_id, "status": "error",
                            "error": str(e), "n_clicks": 0})
        continue

    elapsed = time.time() - t0
    n_true  = sum(1 for r in records if r["label"] == "TRUE_SPOT")
    n_unc   = sum(1 for r in records if r["label"] == "UNCERTAIN")
    print(f"  -> {len(records)} clicks: {n_true} TRUE_SPOT  {n_unc} UNCERTAIN  ({elapsed:.1f}s)")

    all_records.extend(records)
    session_log.append({
        "crop_id": crop_id, "well_id": well_id, "status": "done",
        "n_clicks": len(records), "n_true_spot": n_true,
        "n_uncertain": n_unc, "elapsed_sec": elapsed,
    })

    if records:
        pd.DataFrame(records).to_parquet(
            ANNOTATIONS_DIR / f"{RUN_ID}_{crop_id}_annotations.parquet",
            index=False
        )

print("\n" + "-" * 60)
print(f"Done. Total clicks: {len(all_records)}")

## 8. Consolidate, validate, and export

In [ ]:
all_parquets = sorted(ANNOTATIONS_DIR.glob("*_annotations.parquet"))

if not all_parquets:
    print("No annotations produced.")
    combined_df = pd.DataFrame()
else:
    combined_df = (
        pd.concat([pd.read_parquet(p) for p in all_parquets], ignore_index=True)
        .drop_duplicates(subset=["annotation_id"], keep="last")
        .reset_index(drop=True)
    )
    validate_annotations(combined_df)
    combined_df.to_parquet(canonical_parquet, index=False)
    print(f"Wrote {len(combined_df)} annotations -> {canonical_parquet}")
    display(
        combined_df.groupby(["well_id", "label"])
        .size().rename("n").reset_index()
    )

## 9. Overlay gallery — all channels per crop

In [ ]:
%matplotlib inline

if len(all_parquets) == 0 or len(combined_df) == 0:
    print("No annotations to plot.")
else:
    for crop_id, crop_ann in combined_df.groupby("crop_id"):
        reg_row = crop_registry_df[crop_registry_df["crop_id"] == crop_id]
        if len(reg_row) == 0:
            continue
        reg_row = reg_row.iloc[0]
        key      = (reg_row["dataset_id"], reg_row["well_id"])
        channels = channel_cache.get(key, [])
        if not channels:
            continue

        ymin = int(reg_row["well_ymin_px"])
        xmin = int(reg_row["well_xmin_px"])
        ymax = int(reg_row["well_ymax_px"])
        xmax = int(reg_row["well_xmax_px"])

        n_panels = len(channels)
        fig, axes = plt.subplots(
            1, n_panels,
            figsize=(4 * n_panels, 4),
            gridspec_kw={"wspace": 0.04}
        )
        axes = [axes] if n_panels == 1 else list(axes)

        for ax, ch in zip(axes, channels):
            crop_img = ch["projection"][ymin:ymax, xmin:xmax]
            ax.imshow(
                percentile_clip(crop_img, CFG["display_plo"], CFG["display_phi"]),
                cmap="gray", origin="upper"
            )
            ax.set_title(ch["label"], fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])

            for _, ann in crop_ann.iterrows():
                color = LABEL_COLOR.get(ann["label"], "white")
                # refined position — large cross
                ax.plot(
                    ann["refined_click_crop_x_px"],
                    ann["refined_click_crop_y_px"],
                    marker="+", markersize=14, markeredgewidth=2,
                    color=color, linestyle="none"
                )
                # raw position — small dot
                ax.plot(
                    ann["raw_click_crop_x_px"],
                    ann["raw_click_crop_y_px"],
                    marker=".", markersize=4,
                    color=color, alpha=0.5, linestyle="none"
                )

        n_true = (crop_ann["label"] == "TRUE_SPOT").sum()
        n_unc  = (crop_ann["label"] == "UNCERTAIN").sum()
        fig.suptitle(
            f"{reg_row['well_id']} | {crop_id[-12:]} — "
            f"{n_true} TRUE_SPOT  {n_unc} UNCERTAIN",
            fontsize=9
        )
        plt.tight_layout()
        plt.savefig(GALLERY_DIR / f"{crop_id}_overlay.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close()

    print(f"Saved overlays -> {GALLERY_DIR}")

## 10. Session summary and manifest

In [ ]:
elapsed_total = time.time() - RUN_START_TS

if session_log:
    display(pd.DataFrame(session_log))

manifest = {
    "run_id":                      RUN_ID,
    "notebook_name":               NOTEBOOK_NAME,
    "annotator":                   CFG["annotator"],
    "annotation_registry_version": CFG["annotation_registry_version"],
    "refinement_policy_id":        CFG["refinement_policy_id"],
    "crops_this_session":          len(session_log),
    "clicks_this_session":         len(all_records),
    "total_annotations_on_disk":   len(combined_df) if len(all_parquets) else 0,
    "elapsed_sec":                 elapsed_total,
    "created_at_utc":              datetime.now(timezone.utc).isoformat(),
    "session_log":                 session_log,
}
manifest_path = MANIFEST_DIR / f"{RUN_ID}_run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest -> {manifest_path}")
print(f"Total elapsed: {elapsed_total:.1f}s")

In [ ]:
# -------------------------------------------------------------------
# CONSOLIDATION SAFETY NET
# Run this if Cell 22 (the main consolidation) was skipped or failed.
# Idempotent — safe to run multiple times.
# -------------------------------------------------------------------
from pathlib import Path
import pandas as pd

_canon = TABLES_DIR / "annotations.parquet"
_all_parquets = sorted(ANNOTATIONS_DIR.glob("*_annotations.parquet"))

if not _all_parquets:
    print("No per-crop annotation files found — nothing to consolidate.")
elif _canon.exists():
    _existing = pd.read_parquet(_canon)
    print(f"tables/annotations.parquet already exists ({len(_existing)} rows). Nothing to do.")
    print("If you want to re-consolidate, delete the canonical file and re-run.")
else:
    _combined = (
        pd.concat([pd.read_parquet(p) for p in _all_parquets], ignore_index=True)
        .drop_duplicates(subset=["annotation_id"], keep="last")
        .reset_index(drop=True)
    )
    validate_annotations(_combined)
    _combined.to_parquet(_canon, index=False)
    print(f"Consolidated {len(_all_parquets)} per-crop files -> {len(_combined)} rows -> {_canon}")

# Either way, confirm final state
if _canon.exists():
    _df = pd.read_parquet(_canon)
    display(_df.groupby(["well_id", "crop_id", "label"]).size().rename("n").reset_index())
    
    _missing_from_registry = set(crop_registry_df["crop_id"]) - set(_df["crop_id"].unique())
    if _missing_from_registry:
        print(f"\n[warn] {len(_missing_from_registry)} registered crops have no annotations yet:")
        for _c in sorted(_missing_from_registry):
            print(f"  {_c}")
    else:
        print(f"\nAll {crop_registry_df['crop_id'].nunique()} registered crops have annotations ✓")

## Exit criteria for Notebook 02

- All target crops annotated
- `tables/annotations.parquet` passes §3.6 validation
- Per-crop overlays reviewed across all channels for obvious mis-clicks
- Expected density ~500–1500 TRUE_SPOT per well (§8.1)

**Next:** `03_candidate_universe_generation.ipynb`